# Clasificador base con Iris

Clasificación de las tres especies de Iris con una red neuronal implementada en PyTorch.

## 1. Configuración de PyTorch

Configuración del Entorno Deep Learning: Definición de dispositivos (cpu vs cuda/mps), carga de librerías esenciales (torch, torch.nn, torch.optim) y fijación de semillas de aleatoriedad para reproducibilidad.

In [49]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

In [50]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")


SEED = 42
set_seed(SEED)
device = get_device()

## 2. Preparación del dataset Iris

Los datos se dividen en 80% para entrenamiento y 20% para validación. La normalización se ajusta solamente con el conjunto de entrenamiento.

In [51]:
iris = load_iris()
X = iris.data.astype(np.float32)
y = iris.target.astype(np.int64)

In [52]:
iris.data.shape

(150, 4)

In [53]:
iris.data.dtype

dtype('float64')

In [54]:
iris.target.shape

(150,)

In [55]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [56]:
iris.feature_names

['sepal length (cm)',
 'sepal width (cm)',
 'petal length (cm)',
 'petal width (cm)']

In [57]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

In [58]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)

In [59]:
X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train)
X_val_tensor = torch.from_numpy(X_val)
y_val_tensor = torch.from_numpy(y_val)

In [60]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

In [61]:
batch_size = 16

In [62]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
)

In [63]:
print(f"Ejemplos de entrenamiento: {len(train_dataset)}")
print(f"Ejemplos de validación: {len(val_dataset)}")
print(f"Variables de entrada: {X_train_tensor.shape[1]}")
print(f"Clases: {list(iris.target_names)}")

Ejemplos de entrenamiento: 120
Ejemplos de validación: 30
Variables de entrada: 4
Clases: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


## 3. Arquitectura base

Define una arquitectura nn.Module simple (Clasificador Lineal o MLP).

Se utiliza un MLP con 4 entradas, una capa oculta de 16 neuronas con ReLU y 3 salidas.

In [64]:
class IrisClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, X):
        return self.network(X)

In [65]:
# Definir el modelo
model = IrisClassifier(
    input_dim=X_train_tensor.shape[1],
    hidden_dim=16,
    num_classes=len(iris.target_names),
).to(device)

In [66]:
print(model)
print(f"Dispositivo del modelo: {next(model.parameters()).device}")

IrisClassifier(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=3, bias=True)
  )
)
Dispositivo del modelo: cpu


## 4. Entrenamiento y validación

1) Ciclo de Entrenamiento (Training Loop): Implementación explícita de las fases de forward pass, cálculo de pérdida, backward pass y actualización de pesos.
2) Ciclo de Validación: Evaluación del modelo en datos "no vistos" durante el entrenamiento para detectar overfitting tempranamente.

El entrenamiento ejecuta zero_grad(), forward, cálculo de pérdida, backward() y actualización de parámetros. La validación utiliza model.eval() y torch.no_grad().

In [67]:
# Entrenamiento de una época

def train(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in data_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward
        outputs = model(X_batch)

        # Loss
        loss = criterion(outputs, y_batch)

        # Zero Grad
        optimizer.zero_grad()

        # Backward
        loss.backward()

        # Step
        optimizer.step()

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == y_batch).sum().item()
        total += batch_size

    average_loss = total_loss / total
    accuracy = correct / total
    return average_loss, accuracy

In [68]:
# Evaluación con datos de validación

def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            batch_size = y_batch.size(0)
            total_loss += loss.item() * batch_size
            correct += (outputs.argmax(dim=1) == y_batch).sum().item()
            total += batch_size

    average_loss = total_loss / total
    accuracy = correct / total
    return average_loss, accuracy

## 5. Métricas y tracking

Métricas y Tracking: Registro de la pérdida (loss) y al menos una métrica de desempeño (ej. Accuracy o F1-Score) por cada época.

El modelo se entrena durante 100 épocas con Adam y un learning rate de 0.001. En cada época se registran la pérdida y el accuracy de entrenamiento y validación.

In [69]:
learning_rate = 0.001
epochs = 100
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [70]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

In [71]:
for epoch in range(1, epochs + 1):
    train_loss, train_accuracy = train(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
    )
    val_loss, val_accuracy = evaluate(
        model,
        val_loader,
        criterion,
        device,
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
        print(
            f"Época {epoch:03d} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_accuracy: {train_accuracy:.2%} | "
            f"val_loss: {val_loss:.4f} | "
            f"val_accuracy: {val_accuracy:.2%}"
        )

Época 001 | train_loss: 1.0325 | train_accuracy: 37.50% | val_loss: 1.0121 | val_accuracy: 53.33%
Época 010 | train_loss: 0.7279 | train_accuracy: 86.67% | val_loss: 0.7260 | val_accuracy: 70.00%
Época 020 | train_loss: 0.4802 | train_accuracy: 91.67% | val_loss: 0.5084 | val_accuracy: 80.00%
Época 030 | train_loss: 0.3615 | train_accuracy: 91.67% | val_loss: 0.4054 | val_accuracy: 80.00%
Época 040 | train_loss: 0.2938 | train_accuracy: 91.67% | val_loss: 0.3448 | val_accuracy: 83.33%
Época 050 | train_loss: 0.2466 | train_accuracy: 91.67% | val_loss: 0.2996 | val_accuracy: 86.67%
Época 060 | train_loss: 0.2100 | train_accuracy: 94.17% | val_loss: 0.2642 | val_accuracy: 90.00%
Época 070 | train_loss: 0.1801 | train_accuracy: 95.83% | val_loss: 0.2321 | val_accuracy: 90.00%
Época 080 | train_loss: 0.1564 | train_accuracy: 95.83% | val_loss: 0.2068 | val_accuracy: 93.33%
Época 090 | train_loss: 0.1370 | train_accuracy: 95.83% | val_loss: 0.1845 | val_accuracy: 93.33%
Época 100 | train_lo

## 6. Resultados

Se muestran los valores finales registrados durante el entrenamiento.

In [72]:
print(f"Pérdida inicial de entrenamiento: {history['train_loss'][0]:.4f}")
print(f"Pérdida final de entrenamiento: {history['train_loss'][-1]:.4f}")
print(f"Pérdida final de validación: {history['val_loss'][-1]:.4f}")
print(f"Accuracy final de validación: {history['val_accuracy'][-1]:.2%}")

Pérdida inicial de entrenamiento: 1.0325
Pérdida final de entrenamiento: 0.1216
Pérdida final de validación: 0.1671
Accuracy final de validación: 93.33%


La pérdida de entrenamiento bajó de 1.0325 a 0.1216. En validación, el modelo alcanzó una pérdida de 0.1671 y una accuracy de 93.33%, mostrando un buen desempeño con datos no vistos.